# EduEdge — Gemma 4 Model Exploration

This notebook explores Gemma 4's core capabilities used in EduEdge:
- Text generation with thinking mode
- Multimodal (image) understanding
- Native function calling
- Multilingual responses

**Prerequisite:** Run `bash setup.sh` first to pull the model.

In [ ]:
import ollama
import json
import os
from dotenv import load_dotenv

load_dotenv()

MODEL = os.getenv('GEMMA_MODEL', 'gemma4:e4b')
client = ollama.Client(host=os.getenv('OLLAMA_HOST', 'http://localhost:11434'))

# Verify model is available
models = client.list()
available = [m.model for m in models.models]
print('Available models:', available)
print('Target model available:', MODEL in available or any(MODEL in m for m in available))

## 1. Basic Text — Thinking Mode

In [ ]:
response = client.chat(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': '<|think|>You are a helpful math tutor.'},
        {'role': 'user', 'content': 'If a train travels 120 km in 1.5 hours, what is its average speed?'},
    ],
    options={'temperature': 1.0, 'top_p': 0.95, 'top_k': 64},
)
print(response.message.content)

## 2. Function Calling — Quiz Generation

In [ ]:
import sys
sys.path.insert(0, '..')
from src.tools import TOOLS, dispatch

response = client.chat(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are EduEdge, an AI tutor.'},
        {'role': 'user', 'content': 'Please make me a 3-question quiz on photosynthesis at middle-school level.'},
    ],
    tools=TOOLS,
    options={'temperature': 1.0, 'top_p': 0.95, 'top_k': 64},
)

if response.message.tool_calls:
    for tc in response.message.tool_calls:
        print(f'Tool called: {tc.function.name}')
        print(f'Arguments:   {dict(tc.function.arguments)}')
        result = dispatch(tc.function.name, dict(tc.function.arguments))
        print(f'Result:      {result}')
else:
    print('Direct response:', response.message.content)

## 3. Multilingual Response

In [ ]:
for lang_prompt in [
    '¿Puedes explicar qué es la fotosíntesis en español de forma sencilla?',
    '请用中文简单解释什么是光合作用？',
    'يمكنك شرح التمثيل الضوئي بالعربية؟',
]:
    r = client.chat(
        model=MODEL,
        messages=[{'role': 'user', 'content': lang_prompt}],
        options={'temperature': 1.0, 'top_p': 0.95, 'top_k': 64},
    )
    print(f'Q: {lang_prompt[:60]}...')
    print(f'A: {r.message.content[:200]}...')
    print()

## 4. Multimodal — Image Analysis

Requires an image file. The cell below uses a sample image from `data/`.

In [ ]:
import base64
from pathlib import Path

# Place a test image at data/sample_page.jpg
sample = Path('../data/sample_page.jpg')

if sample.exists():
    with open(sample, 'rb') as f:
        img_b64 = base64.b64encode(f.read()).decode()

    response = client.chat(
        model=MODEL,
        messages=[{
            'role': 'user',
            'content': 'What does this page show? Summarize the key concepts a student should learn from it.',
            'images': [img_b64],
        }],
        options={'temperature': 1.0, 'top_p': 0.95, 'top_k': 64},
    )
    print(response.message.content)
else:
    print('Place a sample image at data/sample_page.jpg to test multimodal.')

## 5. Full EduTutor Session

In [ ]:
from src.tutor import EduTutor

tutor = EduTutor()

turns = [
    'Hi! I have a science exam in 5 days. Can you help me study?',
    'The topics are: photosynthesis, the water cycle, and ecosystems.',
    'Great, give me a quiz on photosynthesis first!',
]

for user_msg in turns:
    print(f'Student: {user_msg}')
    response = ''.join(tutor.chat(user_msg))
    print(f'EduEdge: {response[:400]}...' if len(response) > 400 else f'EduEdge: {response}')
    print()